# Hugging Face NER Project

Final combined notebook: train Hugging Face token-classification models and evaluate them on the project data.


In [2]:
import os
from dataclasses import dataclass
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

BASE_DIR = Path.cwd()
MODEL_CHECKPOINT = "xlm-roberta-base"

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
device

/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'NVIDIA GeForce RTX 2080'

In [3]:
from pathlib import Path

BASE_DIR =Path("/home/ccl/fo41muka/llm-in-medicine")

In [4]:
@dataclass
class NerRun:
    name: str
    train_file: Path
    val_file: Path
    test_file: Path
    model_dir: str
    results_csv: str
    val_split: str = "validation"


ENGLISH = NerRun(
    name="english_i2b2",
    train_file=BASE_DIR / "train.conll",
    val_file=BASE_DIR / "val.conll",
    test_file=BASE_DIR / "test.conll",
    model_dir="xlmr-i2b2-english",
    results_csv="xlmr_english_results.csv",
)

SPANISH = NerRun(
    name="spanish_meddocan",
    train_file=BASE_DIR / "spanish_train.conll",
    val_file=BASE_DIR / "spanish_dev.conll",
    test_file=BASE_DIR / "spanish_test.conll",
    model_dir="xlmr-meddocan-spanish",
    results_csv="xlmr_meddocan_spanish_results.csv",
)

RUNS = [ENGLISH, SPANISH]
for run in RUNS:
    missing = [p for p in (run.train_file, run.val_file, run.test_file) if not p.exists()]
    if missing:
        raise FileNotFoundError(f"{run.name} missing files: {missing}")


In [5]:
def read_conll(path):
    rows, tokens, labels = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            if not parts:
                if tokens:
                    rows.append({"tokens": tokens, "ner_tags": labels})
                    tokens, labels = [], []
                continue
            tokens.append(parts[0])
            labels.append(parts[-1])
    if tokens:
        rows.append({"tokens": tokens, "ner_tags": labels})
    return rows


def label_maps(*splits):
    labels = sorted({label for split in splits for row in split for label in row["ner_tags"]})
    return labels, {label: i for i, label in enumerate(labels)}, dict(enumerate(labels))


def encode_labels(rows, label2id, default=None):
    fallback = None if default is None else label2id[default]
    return [
        {
            "tokens": row["tokens"],
            "ner_tags": [label2id.get(label, fallback) for label in row["ner_tags"]],
        }
        for row in rows
    ]


def dataset_from_splits(**splits):
    return DatasetDict({name: Dataset.from_list(rows) for name, rows in splits.items()})


def tokenizer_fn(tokenizer):
    def tokenize_and_align_labels(examples):
        tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        aligned = []
        for i, labels in enumerate(examples["ner_tags"]):
            previous = None
            ids = []
            for word_id in tokenized.word_ids(batch_index=i):
                ids.append(-100 if word_id is None or word_id == previous else labels[word_id])
                previous = word_id
            aligned.append(ids)
        tokenized["labels"] = aligned
        return tokenized

    return tokenize_and_align_labels


def metric_fn(id2label):
    seqeval = evaluate.load("seqeval")

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred

        if predictions.ndim == 3:
            predictions = np.argmax(predictions, axis=2)
        pairs = [
            (
                [id2label[p] for p, y in zip(pred, gold) if y != -100],
                [id2label[y] for p, y in zip(pred, gold) if y != -100],
            )
            for pred, gold in zip(predictions, labels)
        ]
        pred_labels, true_labels = zip(*pairs)
        result = seqeval.compute(predictions=pred_labels, references=true_labels, mode="strict")
        return {
            "precision": result["overall_precision"],
            "recall": result["overall_recall"],
            "f1": result["overall_f1"],
            "accuracy": result["overall_accuracy"],
        }

    return compute_metrics
    
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim = -1)

In [5]:
def train_and_evaluate(run):
    train_rows = read_conll(run.train_file)
    val_rows = read_conll(run.val_file)
    test_rows = read_conll(run.test_file)
    labels, label2id, id2label = label_maps(train_rows, val_rows, test_rows)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
    tokenize = tokenizer_fn(tokenizer)
    dataset = dataset_from_splits(
        train=encode_labels(train_rows, label2id),
        validation=encode_labels(val_rows, label2id),
        test=encode_labels(test_rows, label2id),
    ).map(tokenize, batched=True)

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=run.model_dir,
            learning_rate=2e-5,
            per_device_train_batch_size=4,
            per_device_eval_batch_size=1,
            num_train_epochs=3,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_steps=50,
            eval_accumulation_steps=16,
            fp16=False,
        ),
        train_dataset=dataset["train"],
        eval_dataset=dataset[run.val_split],
        processing_class=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
        compute_metrics=metric_fn(id2label),
        preprocess_logits_for_metrics= preprocess_logits_for_metrics,
    )

    trainer.train()
    trainer.save_model(run.model_dir)
    tokenizer.save_pretrained(run.model_dir)

    result = trainer.evaluate(dataset["test"])
    pd.DataFrame([result]).to_csv(run.results_csv, index=False)
    return {
        "run": run,
        "trainer": trainer,
        "tokenizer": tokenizer,
        "tokenize": tokenize,
        "label2id": label2id,
        "id2label": id2label,
        "test_result": result,
    }


## Train English model

Trains `xlm-roberta-base` on i2b2 English data and saves `xlmr_english_results.csv`.


In [6]:
english = train_and_evaluate(ENGLISH)
english["test_result"]


Loading weights: 100%|█████████████████████| 197/197 [00:00<00:00, 9786.89it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.024832,0.015637,0.914400,0.917246,0.915821,0.995947
2,0.012243,0.011281,0.955556,0.936836,0.946103,0.997320
3,0.003913,0.011708,0.959377,0.946906,0.953101,0.997647


/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████████████████| 1/1 [00:10<00:00, 10.79s/it]
/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████████████████| 1/1 [00:10<00:00, 10.76s/it]
/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labe

/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.003913,0.014128,3,0.954764,0.946186,0.950456,0.997695


{'eval_loss': 0.014127870090305805,
 'eval_precision': 0.954763597582652,
 'eval_recall': 0.9461863660384006,
 'eval_f1': 0.9504556312483411,
 'eval_accuracy': 0.9976952470497474}

## Test English model on Spanish data

The Spanish labels are mapped to the nearest English i2b2 labels before evaluation.


In [8]:
SPANISH_TO_ENGLISH_ENTITY = {
    "FECHAS": "DATE",
    "EDAD_SUJETO_ASISTENCIA": "AGE",
    "NOMBRE_SUJETO_ASISTENCIA": "PATIENT",
    "NOMBRE_PERSONAL_SANITARIO": "DOCTOR",
    "CORREO_ELECTRONICO": "EMAIL",
    "CALLE": "STREET",
    "TERRITORIO": "CITY",
    "PAIS": "COUNTRY",
    "HOSPITAL": "HOSPITAL",
    "NUMERO_TELEFONO": "PHONE",
    "NUMERO_FAX": "FAX",
    "PROFESION": "PROFESSION",
    "ID_SUJETO_ASISTENCIA": "IDNUM",
    "ID_CONTACTO_ASISTENCIAL": "IDNUM",
    "ID_ASEGURAMIENTO": "IDNUM",
    "ID_TITULACION_PERSONAL_SANITARIO": "IDNUM",
    "ID_EMPLEO_PERSONAL_SANITARIO": "IDNUM",
    "INSTITUCION": "ORGANIZATION",
    "CENTRO_SALUD": "HOSPITAL",
}
SPANISH_TO_ENGLISH_LABEL = {"O": "O"}
SPANISH_TO_ENGLISH_LABEL.update(
    {f"{prefix}-{src}": f"{prefix}-{dst}" for src, dst in SPANISH_TO_ENGLISH_ENTITY.items() for prefix in ("B", "I")}
)


def evaluate_external_conll(artifacts, conll_file, csv_file, label_map=None):
    rows = read_conll(conll_file)
    if label_map:
        encoded = []
        for row in rows:
            encoded.append({
                "tokens": row["tokens"],
                "ner_tags":[
                    artifacts["label2id"][mapped]
                    if (
                        (mapped := label_map.get(label)) is not None
                        and mapped in artifacts["label2id"]
                    )
                    else -100
                    for label in row["ner_tags"]
                ],
            })
    else:
        encoded = encode_labels(
            rows, 
            artifacts["label2id"], 
            default = "0")
    encoded = [
        row
        for row in encoded
        if any(label != -100 for label in row["ner_tags"])
    ]
    dataset = Dataset.from_list(encoded)
    tokenized = dataset.map(artifacts["tokenize"], batched=True)
    result = artifacts["trainer"].evaluate(tokenized)
    pd.DataFrame([result]).to_csv(csv_file, index=False)
    return result


english_on_spanish = evaluate_external_conll(
    english,
    SPANISH.test_file,
    "english_train_spanish_test_results.csv",
    SPANISH_TO_ENGLISH_LABEL,
)
english_on_spanish


Map: 100%|███████████████████████| 8746/8746 [00:00<00:00, 13867.75 examples/s]


/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,1.018370,0,0.446832,0.402778,0.423663,0.947630


{'eval_loss': 1.0183695554733276,
 'eval_precision': 0.4468315972222222,
 'eval_recall': 0.4027777777777778,
 'eval_f1': 0.4236625514403292,
 'eval_accuracy': 0.9476304711899697}

## Train Spanish model

Trains `xlm-roberta-base` on MEDDOCAN Spanish data and saves `xlmr_meddocan_spanish_results.csv`.


In [6]:
spanish = train_and_evaluate(SPANISH)
spanish["test_result"]

Loading weights: 100%|█████████████████████| 197/197 [00:00<00:00, 9306.82it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.072062,0.064849,0.927993,0.948630,0.938198,0.993690
2,0.011751,0.051252,0.961737,0.961903,0.961820,0.995465
3,0.010090,0.046441,0.963461,0.963627,0.963544,0.995812


/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████████████████| 1/1 [00:10<00:00, 10.79s/it]
/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████████████████| 1/1 [00:10<00:00, 10.79s/it]
/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labe

/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.010090,0.055644,3,0.967002,0.968027,0.967514,0.995755


{'eval_loss': 0.05564447119832039,
 'eval_precision': 0.9670019410622904,
 'eval_recall': 0.9680268503797915,
 'eval_f1': 0.9675141242937852,
 'eval_accuracy': 0.9957552649608293}

## Test Spanish model on English data

Only labels with a reliable mapping are compared. Unknown labels are ignored with `-100` during evaluation

In [11]:
ENGLISH_TO_SPANISH_LABEL = {
    "O": "O",
    "B-DATE": "B-FECHAS",
    "I-DATE": "I-FECHAS",
    "B-AGE": "B-EDAD_SUJETO_ASISTENCIA",
    "I-AGE": "I-EDAD_SUJETO_ASISTENCIA",
    "B-PATIENT": "B-NOMBRE_SUJETO_ASISTENCIA",
    "I-PATIENT": "I-NOMBRE_SUJETO_ASISTENCIA",
    "B-DOCTOR": "B-NOMBRE_PERSONAL_SANITARIO",
    "I-DOCTOR": "I-NOMBRE_PERSONAL_SANITARIO",
    "B-EMAIL": "B-CORREO_ELECTRONICO",
    "I-EMAIL": "I-CORREO_ELECTRONICO",
    "B-STREET": "B-CALLE",
    "I-STREET": "I-CALLE",
    "B-CITY": "B-TERRITORIO",
    "I-CITY": "I-TERRITORIO",
    "B-COUNTRY": "B-PAIS",
    "I-COUNTRY": "I-PAIS",
    "B-HOSPITAL": "B-HOSPITAL",
    "I-HOSPITAL": "I-HOSPITAL",
    "B-PHONE": "B-NUMERO_TELEFONO",
    "I-PHONE": "I-NUMERO_TELEFONO",
    "B-FAX": "B-NUMERO_FAX",
    "I-FAX": "I-NUMERO_FAX",
    "B-PROFESSION": "B-PROFESION",
    "I-PROFESSION": "I-PROFESION",
    "B-ORGANIZATION": "B-INSTITUCION",
    "I-ORGANIZATION": "I-INSTITUCION",
}

spanish_on_english = evaluate_external_conll(
    spanish,
    ENGLISH.test_file,
    "spanish_train_english_test_results.csv",
    ENGLISH_TO_SPANISH_LABEL,
)
spanish_on_english

Map: 100%|█████████████████████| 23778/23778 [00:01<00:00, 20782.68 examples/s]


/home/ccl/fo41muka/llm-in-medicine/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.213102,0,0.385077,0.267850,0.315940,0.965277


{'eval_loss': 0.21310238540172577,
 'eval_precision': 0.38507670850767084,
 'eval_recall': 0.26785021342646487,
 'eval_f1': 0.3159400389060533,
 'eval_accuracy': 0.965277475042504}